# 10_c - Match DBLP papers to OpenAlex (key-to-anonymous fallback)

**Purpose:** Continue the paper-level DBLP-to-OpenAlex matching pipeline while using an OpenAlex API key only while its daily allowance is available. When the keyed allowance returns HTTP 429, this notebook switches all subsequent requests in the current session to anonymous access.

**Important:** Anonymous access has its own, smaller daily limit. When that limit returns HTTP 429, the notebook saves its checkpoint and stops cleanly. It can be resumed after the reset.

**Scope:** Paper-level matching only. It does not build author-level records or assign junior/senior status.

**Inputs:** `output/dblp_papers_raw.csv` and, if it exists, `output/dblp_openalex_checkpoint.csv`.

**Outputs:** matched, ambiguous, unmatched, coverage-report, and checkpoint CSV files in `output/`.

In [1]:
import os, re, json, time, unicodedata
from pathlib import Path
from difflib import SequenceMatcher

import pandas as pd
import requests

INPUT_PATH = Path("output/dblp_papers_raw.csv")
OUTPUT_DIR = Path("output")
CHECKPOINT_PATH = OUTPUT_DIR / "dblp_openalex_checkpoint.csv"
MAILTO = os.getenv("OPENALEX_MAILTO", "")
API_KEY = "G0ErBTn3I9KHeI8oYZv0pd"  # Leave blank to begin anonymously.
USE_API_KEY = bool(API_KEY and API_KEY != "PASTE_YOUR_OPENALEX_API_KEY_HERE")
API_URL = "https://api.openalex.org/works"
PER_PAGE = 10
SLEEP_SECONDS = 0.12
YEAR_TOLERANCE = 1
AUTO_ACCEPT_SCORE = 0.94
AMBIGUOUS_SCORE = 0.82

OUTPUT_DIR.mkdir(exist_ok=True)
assert INPUT_PATH.exists(), f"Missing {INPUT_PATH}. Run Notebook 09 first."
print(f"Input: {INPUT_PATH}")
print("Starting mode:", "API key" if USE_API_KEY else "anonymous")

Input: output\dblp_papers_raw.csv
Starting mode: API key


## Load and audit input

The notebook expects at least `conference`, `year`, `title`, and `authors`. It creates a stable `dblp_row_id` so matching can be restarted safely.

In [2]:
papers = pd.read_csv(INPUT_PATH)
required = {"conference", "year", "title", "authors"}
missing = required - set(papers.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

papers = papers.copy()
papers["conference"] = papers["conference"].astype(str).str.upper().str.strip()
papers["year"] = pd.to_numeric(papers["year"], errors="coerce").astype("Int64")
papers["title"] = papers["title"].fillna("").astype(str).str.strip()
papers["authors"] = papers["authors"].fillna("").astype(str).str.strip()
papers = papers.loc[papers["title"].ne("") & papers["year"].notna()].reset_index(drop=True)
papers["dblp_row_id"] = [f"DBLP_{i:07d}" for i in range(1, len(papers) + 1)]

print(f"Papers available: {len(papers):,}")
print(f"Duplicate conference-year-title rows: {papers.duplicated(['conference','year','title']).sum():,}")
papers.groupby("conference").size().sort_values(ascending=False).head(10)

Papers available: 83,319
Duplicate conference-year-title rows: 36


conference
CVPR       8458
AAAI       7004
NIPS       6502
IJCAI      5612
CHI        5522
INFOCOM    5499
CIKM       4477
ICML       4106
SIGIR      3501
KDD        3026
dtype: int64

## Matching functions

Candidates come from OpenAlex title search. A candidate is scored from normalized-title similarity, year agreement, and surname overlap. Candidate metadata and scoring fields are retained for auditability.

In [3]:
class OpenAlexQuotaError(RuntimeError):
    """Raised only after both keyed and anonymous access return HTTP 429."""


def normalize_text(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower().replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def title_similarity(left, right):
    return SequenceMatcher(None, normalize_text(left), normalize_text(right)).ratio()


def surname_set(authors):
    names = re.split(r"\s*(?:,|;| and )\s*", str(authors))
    result = set()
    for name in names:
        tokens = normalize_text(name).split()
        if tokens:
            result.add(tokens[-1])
    return result


def work_surnames(work):
    names = []
    for authorship in work.get("authorships", []) or []:
        name = ((authorship.get("author") or {}).get("display_name"))
        if name:
            names.append(name)
    return surname_set("; ".join(names))


def year_score(dblp_year, work_year):
    if work_year is None:
        return 0.0
    gap = abs(int(dblp_year) - int(work_year))
    return 1.0 if gap == 0 else 0.75 if gap <= YEAR_TOLERANCE else 0.0


def candidate_score(row, work):
    title_score = title_similarity(row.title, work.get("title") or "")
    y_score = year_score(row.year, work.get("publication_year"))
    db_surnames, oa_surnames = surname_set(row.authors), work_surnames(work)
    overlap = len(db_surnames & oa_surnames) / len(db_surnames) if db_surnames else 0.0
    total = 0.78 * title_score + 0.14 * y_score + 0.08 * overlap
    return {"match_score": round(total, 4), "title_score": round(title_score, 4),
            "year_score": round(y_score, 4), "author_overlap": round(overlap, 4)}


def clean_search_query(text, max_length=300):
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"[*?:;'\"]", " ", text)
    return re.sub(r"\s+", " ", text).strip()[:max_length]


def fetch_candidates(query, retries=3):
    """Use key if available; permanently switch this session to anonymous after keyed 429."""
    global USE_API_KEY
    safe_query = clean_search_query(query)
    last_error = None

    for attempt in range(retries):
        params = {"search": safe_query, "per-page": PER_PAGE,
                  "select": "id,doi,title,display_name,publication_date,publication_year,cited_by_count,authorships,primary_location,type"}
        if MAILTO:
            params["mailto"] = MAILTO
        if USE_API_KEY:
            params["api_key"] = API_KEY

        try:
            response = requests.get(API_URL, params=params, timeout=45)
            if response.status_code == 429:
                if USE_API_KEY:
                    USE_API_KEY = False
                    print("Keyed OpenAlex allowance exhausted; switching to anonymous access.")
                    continue
                retry_after = response.headers.get("Retry-After", "unknown")
                raise OpenAlexQuotaError(
                    f"Anonymous OpenAlex allowance is exhausted (Retry-After: {retry_after})."
                )
            response.raise_for_status()
            time.sleep(SLEEP_SECONDS)
            return response.json().get("results", [])
        except OpenAlexQuotaError:
            raise
        except requests.RequestException as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(2 ** attempt)

    raise last_error


def fallback_query(row):
    return " ".join([normalize_text(row.title), *sorted(surname_set(row.authors))[:3]])


def get_scored_candidates(row):
    title_candidates = fetch_candidates(row.title)
    scored = [(work, candidate_score(row, work)) for work in title_candidates]
    scored.sort(key=lambda item: item[1]["match_score"], reverse=True)
    if (scored[0][1]["match_score"] if scored else 0.0) < AMBIGUOUS_SCORE:
        try:
            fallback_candidates = fetch_candidates(fallback_query(row))
            works = {work.get("id"): work for work, _ in scored if work.get("id")}
            works.update({work.get("id"): work for work in fallback_candidates if work.get("id")})
            scored = [(work, candidate_score(row, work)) for work in works.values()]
            scored.sort(key=lambda item: item[1]["match_score"], reverse=True)
        except OpenAlexQuotaError:
            raise
        except requests.RequestException:
            pass
    return scored


def flatten_result(row, work, score, status, rank):
    primary_source = (((work.get("primary_location") or {}).get("source") or {}).get("display_name"))
    row_dict = row._asdict() if hasattr(row, "_asdict") else row.to_dict()
    return {**row_dict, **score, "match_status": status, "candidate_rank": rank,
            "openalex_id": work.get("id"), "openalex_doi": work.get("doi"),
            "openalex_title": work.get("title") or work.get("display_name"),
            "openalex_publication_date": work.get("publication_date"),
            "openalex_publication_year": work.get("publication_year"),
            "openalex_cited_by_count": work.get("cited_by_count"),
            "openalex_primary_source": primary_source, "openalex_type": work.get("type"),
            "openalex_authorships": json.dumps(work.get("authorships", []), ensure_ascii=False)}

## Resume-aware matching run

Begin with `MAX_ROWS = 100` to validate matches, inspect the ambiguous output, then set it to `None` for the full run. Previously checkpointed rows are skipped.

In [4]:
MAX_ROWS = None  # Use a small number first only if you want another validation batch.  # Keep at 100 for validation; set to None for the full corpus.

existing = pd.read_csv(CHECKPOINT_PATH) if CHECKPOINT_PATH.exists() else pd.DataFrame()
# Keep accepted matches; retry prior unresolved/API-error rows with the new fallback logic.
accepted_ids = set(existing.loc[existing["match_status"].eq("matched"), "dblp_row_id"].astype(str)) if not existing.empty else set()
todo = papers.loc[~papers["dblp_row_id"].isin(accepted_ids)].copy()
if MAX_ROWS is not None:
    todo = todo.head(MAX_ROWS)

new_rows = []
for position, row in enumerate(todo.itertuples(index=False), start=1):
    try:
        scored = get_scored_candidates(row)
        if not scored:
            new_rows.append({**row._asdict(), "match_status":"unmatched", "match_score":0.0,
                             "title_score":0.0, "year_score":0.0, "author_overlap":0.0,
                             "candidate_rank":None, "openalex_id":None, "openalex_doi":None,
                             "openalex_title":None, "openalex_publication_date":None,
                             "openalex_publication_year":None, "openalex_cited_by_count":None,
                             "openalex_primary_source":None, "openalex_type":None,
                             "openalex_authorships":None, "error":None})
        else:
            best_work, best_score = scored[0]
            margin = best_score["match_score"] - (scored[1][1]["match_score"] if len(scored) > 1 else 0)
            status = "matched" if best_score["match_score"] >= AUTO_ACCEPT_SCORE and margin >= 0.03 else "ambiguous" if best_score["match_score"] >= AMBIGUOUS_SCORE else "unmatched"
            new_rows.append(flatten_result(row, best_work, best_score, status, 1))
        if position % 25 == 0:
            print(f"Processed {position:,}/{len(todo):,}")
    except OpenAlexQuotaError as exc:
        new_results = pd.DataFrame(new_rows)
        updated = pd.concat([
            existing.loc[~existing["dblp_row_id"].isin(new_results["dblp_row_id"])],
            new_results
        ], ignore_index=True) if not existing.empty else new_results
        updated.to_csv(CHECKPOINT_PATH, index=False)
        print(f"\nStopped safely: {exc}")
        print(f"Checkpoint saved with {len(updated):,} rows.")
        raise
    except requests.RequestException as exc:
        new_rows.append({**row._asdict(), "match_status":"api_error", "match_score":None,
                         "title_score":None, "year_score":None, "author_overlap":None,
                         "candidate_rank":None, "openalex_id":None, "openalex_doi":None,
                         "openalex_title":None, "openalex_publication_date":None,
                         "openalex_publication_year":None, "openalex_cited_by_count":None,
                         "openalex_primary_source":None, "openalex_type":None,
                         "openalex_authorships":None, "error":str(exc)})

# New attempts replace older unresolved rows rather than creating duplicate paper records.
new_results = pd.DataFrame(new_rows)
if existing.empty:
    updated = new_results
else:
    updated = pd.concat([
        existing.loc[~existing["dblp_row_id"].isin(new_results["dblp_row_id"])],
        new_results
    ], ignore_index=True)
updated.to_csv(CHECKPOINT_PATH, index=False)
print(f"Checkpoint rows: {len(updated):,}; retried/newly processed: {len(new_rows):,}")

Keyed OpenAlex allowance exhausted; switching to anonymous access.
Processed 25/83,075
Processed 50/83,075

Stopped safely: Anonymous OpenAlex allowance is exhausted (Retry-After: 17027).
Checkpoint saved with 317 rows.


OpenAlexQuotaError: Anonymous OpenAlex allowance is exhausted (Retry-After: 17027).

## Write final outputs and coverage report

Only `matched` rows are treated as accepted matches. Ambiguous rows remain separate for review; they are not silently included in later analysis.

In [ ]:
results = pd.read_csv(CHECKPOINT_PATH)
matched = results.loc[results.match_status.eq("matched")].copy()
ambiguous = results.loc[results.match_status.eq("ambiguous")].copy()
unmatched = results.loc[results.match_status.isin(["unmatched", "api_error"])].copy()

matched.to_csv(OUTPUT_DIR / "dblp_openalex_matched.csv", index=False)
ambiguous.to_csv(OUTPUT_DIR / "dblp_openalex_ambiguous.csv", index=False)
unmatched.to_csv(OUTPUT_DIR / "dblp_openalex_unmatched.csv", index=False)

report = (results.assign(processed=1, matched_flag=results.match_status.eq("matched").astype(int))
          .groupby(["conference", "year"], dropna=False)
          .agg(papers=("processed", "sum"), matched=("matched_flag", "sum"),
               ambiguous=("match_status", lambda s: (s == "ambiguous").sum()),
               unmatched=("match_status", lambda s: s.isin(["unmatched", "api_error"]).sum()),
               mean_match_score=("match_score", "mean"))
          .reset_index())
report["match_rate"] = report["matched"] / report["papers"]
report.to_csv(OUTPUT_DIR / "dblp_openalex_match_report.csv", index=False)

print(results.match_status.value_counts(dropna=False))
print(f"Accepted match rate among processed rows: {len(matched)/len(results):.1%}" if len(results) else "No rows processed.")
report.sort_values(["match_rate", "papers"]).head(15)

## Manual review checklist

Before proceeding to author extraction in Notebook 11:
- Inspect a random sample of accepted matches, especially short or generic titles.
- Inspect all ambiguous rows with high scores.
- Re-run this notebook with `MAX_ROWS = None` only after the scoring thresholds look reliable.
- Keep the checkpoint and all three result files as the paper-level matching audit trail.